In [1]:
%pip install -q langchain langchain-google-genai langchain-community google-generativeai


In [2]:
import langchain
import langchain_google_genai
import langchain_community

/tmp/ipykernel_3467/2589394304.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [4]:
import sqlite3

DB_PATH = 'students.db'

def create_database():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute('DROP TABLE IF EXISTS students')
    cur.execute('''
        CREATE TABLE students (
            student_id  TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            department  TEXT NOT NULL,
            python      INTEGER NOT NULL,
            database_m  INTEGER NOT NULL,
            ai          INTEGER NOT NULL,
            web         INTEGER NOT NULL
        )
    ''')
    rows = [
        ('22CS045', 'Dhanushya', 'Computer Science',       85, 72, 90, 78),
        ('22CS046', 'Rahul',     'Computer Science',       65, 70, 68, 72),
        ('22CS047', 'Priya',     'Information Technology', 92, 88, 95, 90),
        ('22CS048', 'Arun',      'Information Technology', 55, 60, 58, 62),
        ('22CS049', 'Meena',     'Computer Science',       78, 85, 80, 88),
    ]
    cur.executemany('INSERT INTO students VALUES (?,?,?,?,?,?,?)', rows)
    conn.commit()
    conn.close()
    print(f'Database created with {len(rows)} records ✓')

create_database()


Database created with 5 records ✓


## Step 4 — Define the four LangChain tools


In [5]:
import sqlite3
from langchain.tools import tool


@tool
def get_student_info(student_id: str) -> str:
    """Return the name and department of a student given their student_id.
    Use this when the user asks about a student's personal details."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute('SELECT name, department FROM students WHERE student_id=?',
                (student_id.strip(),))
    row = cur.fetchone()
    conn.close()
    if row:
        return f'Name: {row[0]}, Department: {row[1]}'
    return f'Student {student_id} not found.'


@tool
def get_student_marks(student_id: str) -> str:
    """Return subject-wise marks (Python, Database, AI, Web) for a student.
    Use this when the user asks about marks or when you need marks for
    computing totals, averages, or pass/fail status."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute('SELECT python, database_m, ai, web FROM students WHERE student_id=?',
                (student_id.strip(),))
    row = cur.fetchone()
    conn.close()
    if row:
        return (f'Python: {row[0]}/100, Database: {row[1]}/100, '
                f'AI: {row[2]}/100, Web: {row[3]}/100')
    return f'Student {student_id} not found.'


@tool
def calculator(expression: str) -> str:
    """Evaluate a safe arithmetic expression and return the result.
    Use this to compute totals, averages, or percentages.
    Example expressions: '85+72+90+78', '(85+72+90+78)/4'"""
    try:
        safe = {'__builtins__': {}, 'sum': sum, 'round': round,
                'len': len, 'min': min, 'max': max, 'abs': abs}
        return str(eval(expression, safe))  # noqa: S307
    except Exception as e:
        return f'Calculator error: {e}'


@tool
def get_passing_rules() -> str:
    """Return the university passing rules.
    Use this when the user asks if a student has passed or is eligible."""
    return (
        'University Passing Rules:\n'
        '1. Minimum overall average: 40 out of 100.\n'
        '2. Minimum mark in EACH subject: 35 out of 100.\n'
        'Both conditions must be satisfied to pass.'
    )


tools = [get_student_info, get_student_marks, calculator, get_passing_rules]
print('Tools ready:', [t.name for t in tools])


Tools ready: ['get_student_info', 'get_student_marks', 'calculator', 'get_passing_rules']


In [6]:
import langchain

print(langchain.__version__)

1.3.18


In [7]:
!pip show langchain

Name: langchain
Version: 1.3.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [8]:
import langchain
import langchain_core
import langchain_google_genai

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("Google GenAI:", langchain_google_genai.__version__)

LangChain: 1.3.18
LangChain Core: 1.6.1
Google GenAI: 4.4.0


In [11]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

In [13]:
%pip install -q langchain-classic

In [12]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY,
)

prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful student academic assistant for SJIT college. '
     'Always use the appropriate tools to look up real data. '
     'Do NOT guess marks, names, or rules — call a tool to get them. '
     'When multiple pieces of information are needed, call tools one at a time.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
)
print('Agent ready ✓')


Agent ready ✓


In [18]:
def ask(question):
    print('\n' + '='*65)
    print('QUESTION:', question)
    print('='*65)
    result = agent_executor.invoke({'input': question})
    print('\n--- FINAL ANSWER ---')
    print(result['output'])
    return result['output']


## Q1 — Name and department  (`get_student_info`)


In [21]:
ask('What is the name and department of student 22CS045?')



QUESTION: What is the name and department of student 22CS045?


> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_student_info` with `{'student_id': '22CS045'}`


Name: Dhanushya, Department: Computer Science

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The student with ID **22CS045** is **Dhanushya** from the **Computer Science** department.', 'index': 0, 'extras': {'signature': 'EoICCv8BAWkUfRPBEqBkkmV3l/XJIaOyiczjWOrW7Y9jQcevItLKRz0idFp0nmn0g6TUcv1OZjvI2MOdP2KXnAO7vkIG4xfndmckQIo5RWvHFQ3olXZ/aJseL4Fif3iSZa6vk7vS34wbr44fEEdW9k1+RqyRHDR6TJMB9NtCn4+dDQOXlbwFm2MA8yFu1YcvZjtmMZa13U4covy2bsHQCKY6t9OiBg6SB3uLwEuwKk/jZPeUX2tjqXJPXCyOOHB1dCyGhr89S7PnOgB8qxv8WzkPeiqWUktyeWjd6WHSq0XOAEo8lXyQe8CIIF14HBISJrLFfT75lDzlYfIA40tVYwDI3mnO'}}]

> Finished chain.

--- FINAL ANSWER ---
[{'type': 'text', 'text': 'The student with ID **22CS045** is **Dhanushya** from the **Computer Science** department.', 'index': 0, 'extras': {'signature': 'EoICCv8BAWkUfRPBEqBkkmV3l/XJIaOyiczjWOrW7Y9jQcevItLKRz0idFp0nmn0g6TUcv1OZjvI2MOdP2KXnAO7vkIG4xfndmckQIo5RWvHFQ3olXZ/aJseL4Fif3iSZa6vk7vS34wbr44fEEdW9k1+RqyRHDR6TJMB9NtCn4+dDQOXlbwFm2MA8yFu1YcvZjtmMZa13U4covy2bsHQCKY6t9OiBg6SB3uLwEuwKk/jZPeUX2tjqXJPXCyOOHB1dCyGhr89S7PnOgB8qxv8WzkPeiqWUktyeWjd

[{'type': 'text',
  'text': 'The student with ID **22CS045** is **Dhanushya** from the **Computer Science** department.',
  'index': 0,
  'extras': {'signature': 'EoICCv8BAWkUfRPBEqBkkmV3l/XJIaOyiczjWOrW7Y9jQcevItLKRz0idFp0nmn0g6TUcv1OZjvI2MOdP2KXnAO7vkIG4xfndmckQIo5RWvHFQ3olXZ/aJseL4Fif3iSZa6vk7vS34wbr44fEEdW9k1+RqyRHDR6TJMB9NtCn4+dDQOXlbwFm2MA8yFu1YcvZjtmMZa13U4covy2bsHQCKY6t9OiBg6SB3uLwEuwKk/jZPeUX2tjqXJPXCyOOHB1dCyGhr89S7PnOgB8qxv8WzkPeiqWUktyeWjd6WHSq0XOAEo8lXyQe8CIIF14HBISJrLFfT75lDzlYfIA40tVYwDI3mnO'}}]

## Q2 — Subject marks  (`get_student_marks`)


In [ ]:
ask('What are the marks of 22CS047?')


## Q3 — Total and average  (`get_student_marks` → `calculator`)


In [ ]:
ask('What is the total and average mark of 22CS045?')


## Q4 — Pass eligibility  (`get_student_marks` → `get_passing_rules` → `calculator`)


In [ ]:
ask('Is 22CS045 eligible to pass according to the university rules?')


## 🏆 Challenge Question — All four tools


In [22]:
ask(
    'I am 22CS045. Tell me my name, department, total marks, '
    'average marks, and whether I satisfy the university passing requirements.'
)



QUESTION: I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements.


> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_student_info` with `{'student_id': '22CS045'}`


Name: Dhanushya, Department: Computer Science

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_student_marks` with `{'student_id': '22CS045'}`


Python: 85/100, Database: 72/100, AI: 90/100, Web: 78/100

...

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_passing_rules` with `{}`


University Passing Rules:
1. Minimum overall average: 40 out of 100.
2. Minimum mark in EACH subject: 35 out of 100.
Both conditions must be satisfied to pass.

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here are your details, **Dhanushya** (Student ID: **22CS045**):\n\n### Personal Information\n* **Name:** Dhanushya\n* **Department:** Computer Science\n\n---\n\n### Academic Performance\n* **Python:** 85 / 100\n* **Database:** 72 / 100\n* **AI:** 90 / 100\n* **Web:** 78 / 100\n\n* **Total Marks:** **325** / 400\n* **Average Marks:** **81.25%**\n\n---\n\n### Passing Status\n* **University Passing Rules:**\n  1. Minimum score of **35/100** in each subject.\n  2. Minimum overall average of **40/100**.\n\n**Result:** **PASSED**  \nYou meet all university requirements (you scored at least 35 in every subject and your average of 81.25 exceeds the 40 mark threshold).', 'index': 0, 'extras': {'signature': 'ErMGCrAGAWkUfROXgpEz34TVM7F6EQixEysrJvoHswEYvPXTnmhgNP/3LJ2TLs7NH8Oy3cQLRtQkK0RS4r9vhU4blGGtijVyWdvQim6CpFv7ky/MVXj5l9dNO5FV5MenePK5I51Vzo0RlaEB0a6db/9VDCphKzusPwv8O3w9ZHYnvvA4yk+dDLyBwzwLrtoMU6q3P8qHq9A+cN6dutrZPLigP1Gi1BOZ6FnRvaGQU5jJUFPrLUmZr5SWdzZthLHxFfeCwixp1

[{'type': 'text',
  'text': 'Here are your details, **Dhanushya** (Student ID: **22CS045**):\n\n### Personal Information\n* **Name:** Dhanushya\n* **Department:** Computer Science\n\n---\n\n### Academic Performance\n* **Python:** 85 / 100\n* **Database:** 72 / 100\n* **AI:** 90 / 100\n* **Web:** 78 / 100\n\n* **Total Marks:** **325** / 400\n* **Average Marks:** **81.25%**\n\n---\n\n### Passing Status\n* **University Passing Rules:**\n  1. Minimum score of **35/100** in each subject.\n  2. Minimum overall average of **40/100**.\n\n**Result:** **PASSED**  \nYou meet all university requirements (you scored at least 35 in every subject and your average of 81.25 exceeds the 40 mark threshold).',
  'index': 0,
  'extras': {'signature': 'ErMGCrAGAWkUfROXgpEz34TVM7F6EQixEysrJvoHswEYvPXTnmhgNP/3LJ2TLs7NH8Oy3cQLRtQkK0RS4r9vhU4blGGtijVyWdvQim6CpFv7ky/MVXj5l9dNO5FV5MenePK5I51Vzo0RlaEB0a6db/9VDCphKzusPwv8O3w9ZHYnvvA4yk+dDLyBwzwLrtoMU6q3P8qHq9A+cN6dutrZPLigP1Gi1BOZ6FnRvaGQU5jJUFPrLUmZr5SWdzZthLHxFfe

## Bonus — Interactive chat loop
Type any question about any student. Type `quit` to stop.


In [ ]:
print('Available IDs: 22CS045, 22CS046, 22CS047, 22CS048, 22CS049')
while True:
    q = input('You: ').strip()
    if not q or q.lower() in ('quit','exit','q'):
        print('Goodbye!')
        break
    ask(q)
